# Starter: an agent over a folder of your files

Point RI #5's tool server at any folder of text files and run two agents on your own questions: `C5`, which has `grep` and `open` and no index, and `C0`, which has dense `search` only. The folder here is this repository's `docs/`, so the notebook runs anywhere. Change `FOLDER`, `PATTERNS`, and `QUESTIONS` to use yours.

`grep` runs ripgrep over the files on disk, the way a coding agent searches a repository: a change to a file shows up in the next grep with no index to rebuild. When `rg` is not installed, the notebook falls back to the Python regular-expression backend over the same files and says so.

Agent runs call a model, so they sit behind `RUN_EXPENSIVE`. With it off, the cells show the tools by hand and read any run you made earlier from the cache.

In [1]:
import os
import shutil
from pathlib import Path

from agent_arms import ENCODER, Question, run_arm, tool_server
from aieng.advanced_rag import spend
from aieng.advanced_rag.agents.corpus_tools import STANDARD_ARMS, Corpus, DenseSearch, RegexGrep, RipgrepGrep
from aieng.advanced_rag.models import choose_models
from aieng.advanced_rag.retrieval.embedders import SentenceTransformerEmbedder
from aieng.advanced_rag.runs import ReferenceMissingError


os.environ.setdefault("HF_HUB_OFFLINE", "1")

# Agent runs call your model. Leave False to see the tools and any cached run.
RUN_EXPENSIVE = False
MODELS = choose_models(show=True)

FOLDER = Path.cwd().parents[1] / "docs"
PATTERNS = ("**/*.md", "**/*.txt", "**/*.rst", "**/*.py")
DESCRIPTION = "the documentation of a Python package for retrieval experiments"

family google
  generation  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  judging     gemini-3.8-flash        $0.75 in / $3.75 out per 1M tokens; introductory price
  extraction  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  embeddings  BAAI/bge-small-en-v1.5  local
key budget not set on the key; assuming $300.00 (set ADVANCED_RAG_KEY_BUDGET_USD to override), spent $135.44, left $164.56


## Load the folder and pick a grep backend

`Corpus.from_folder` reads one document per file, with its path relative to the folder as the identifier. Large files are skipped (2 MB by default).

In [2]:
corpus = Corpus.from_folder(FOLDER, patterns=PATTERNS, name=f"folder-{FOLDER.name}")
print(f"{len(corpus)} files, {corpus.characters:,} characters, about {corpus.characters // 4:,} tokens")

if shutil.which("rg"):
    grep_backend = RipgrepGrep(FOLDER, corpus=corpus)
    print("grep: ripgrep over the files on disk")
else:
    grep_backend = RegexGrep(corpus)
    print("grep: rg is not installed, so Python regular expressions over the files in memory")

3 files, 40,646 characters, about 10,161 tokens
grep: rg is not installed, so Python regular expressions over the files in memory


## Call the tools by hand

This is what the agent sees. `grep` returns the matching files with the first matching line and a preview, plus the total number of files that match. `open` returns numbered lines, capped at 4,000 tokens with a note naming the line to continue from.

In [3]:
server = tool_server(corpus, grep=grep_backend)
session = server.session(STANDARD_ARMS["C5"])
found = session.call("grep", {"pattern": "SPEND_LIMIT"})
print("total_hits:", found["total_hits"])
for match in found["matches"]:
    print(f"  {match['doc_id']}:{match['line']}  {match['preview'][:110]}")

total_hits: 1
  models-and-spend.md:61  ...odel. | | `ADVANCED_RAG_EXTRACTION_MODEL` | Overrides the extraction model. | | `ADVANCED_RAG_SPEND_LIMIT_U


In [4]:
first = found["matches"][0]
opened = session.call(
    "open", {"doc_id": first["doc_id"], "start_line": first["line"] - 2, "end_line": first["line"] + 2}
)
print(opened["text"])
print()
print("trace:", [(e.name, e.arguments) for e in session.trace.events])

59: | `ADVANCED_RAG_JUDGE_MODEL` | Overrides the judging model. |
60: | `ADVANCED_RAG_EXTRACTION_MODEL` | Overrides the extraction model. |
61: | `ADVANCED_RAG_SPEND_LIMIT_USD` | Stops a run once its estimate passes this many dollars. Unset means no limit beyond the key's budget. |
62: | `ADVANCED_RAG_KEY_BUDGET_USD` | The budget, in US dollars, the spend guards assume when your key reports none. Default 300. Ignored when the key reports a budget. |
63: 

trace: [('grep', {'pattern': 'SPEND_LIMIT', 'ignore_case': True}), ('open', {'doc_id': 'models-and-spend.md', 'start_line': 59, 'end_line': 63})]


## Your questions

Write each question with the answer you expect, so the run can score itself. The answer is optional; without one, the table shows the agent's answers to read.

In [5]:
QUESTIONS = [
    Question(
        "q1",
        "Which environment variable stops a run once its estimated cost passes a dollar amount?",
        "ADVANCED_RAG_SPEND_LIMIT_USD",
    ),
    Question(
        "q2",
        "Which package do I install when extracting the ServerFault dump fails because 7z is missing?",
        "p7zip-full",
    ),
    Question("q3", "Which embedding model do all model families use?", "BAAI/bge-small-en-v1.5"),
    Question("q4", "What command fetches the core corpora?", "python -m aieng.advanced_rag.corpora.download --core"),
    Question("q5", "Who shares one judge model, and which model is it?", "gemini-3.8-flash"),
]

## Run `C5` and `C0`

Each run is cached by arm, model, folder contents, and questions. The first run with `RUN_EXPENSIVE = True` calls your generation model, up to 30 calls a question. `C0` embeds every file with the local `bge-small-en-v1.5` model first; the embedding reads only the first 512 tokens of a file, so long files are better split before you rely on `C0`.

In [6]:
def search():
    """Embed every file, the first time C0 runs live."""
    return DenseSearch(corpus, SentenceTransformerEmbedder(ENCODER))


results = {}
for arm in ("C5", "C0"):
    try:
        results[arm] = run_arm(
            arm,
            QUESTIONS,
            corpus=corpus,
            label=corpus.name,
            models=MODELS,
            live=RUN_EXPENSIVE,
            search=search,
            grep=grep_backend,
            description=DESCRIPTION,
        )
    except ReferenceMissingError:
        print(f"{arm}: no cached run for this folder and model; set RUN_EXPENSIVE = True to run it")

agent-runs reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
           23 calls, 45,210 prompt + 827 completion tokens
           rebuilding with gemini-3.5-flash-lite: about 23 calls, est. $0.02 (set RUN_EXPENSIVE = True to rebuild)
agent-runs reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
           51 calls, 984,077 prompt + 2,238 completion tokens
           rebuilding with gemini-3.5-flash-lite: about 51 calls, est. $0.30 (set RUN_EXPENSIVE = True to rebuild)


In [7]:
for arm, result in results.items():
    kind = "reference trajectories" if result.origin == "reference" else "your runs"
    print(f"{arm}, {kind} on {result.value['model']}")
    for run in result.value["runs"]:
        cited = ", ".join(run["read_ids"][:3]) or "nothing opened"
        print(
            f"  {run['question_id']}  F1 {run['f1']:.2f}  {run['model_calls']:>2} calls  {run['answer'][:60]!r}  [{cited}]"
        )

C5, reference trajectories on Qwen3_6-35B-A3B
  q1  F1 1.00   5 calls  'ADVANCED_RAG_SPEND_LIMIT_USD'  [models-and-spend.md]
  q2  F1 0.25   4 calls  'p7zip-full (Debian and Ubuntu) or p7zip (Homebrew)'  [setup.md]
  q3  F1 1.00   6 calls  'BAAI/bge-small-en-v1.5'  [models-and-spend.md]
  q4  F1 0.80   3 calls  '`uv run python -m aieng.advanced_rag.corpora.download --core'  [setup.md]
  q5  F1 0.40   5 calls  'All families share gemini-3.8-flash'  [models-and-spend.md]
C0, reference trajectories on Qwen3_6-35B-A3B
  q1  F1 0.00  30 calls  'ADVANCED_RAG_MAX_COST'  [nothing opened]
  q2  F1 0.00  15 calls  'p7zip'  [nothing opened]
  q3  F1 1.00   2 calls  'BAAI/bge-small-en-v1.5'  [nothing opened]
  q4  F1 0.00   2 calls  'bash scripts/setup.sh'  [nothing opened]
  q5  F1 0.29   2 calls  'Every family shares the gemini-3.8-flash judge model.'  [nothing opened]


In [8]:
print(spend.report())

this run, estimated   est. $0.00 over 0 models
this run, proxy usage $0.00 (team key: includes teammates' calls in the same window)
key budget not set on the key; assuming $300.00 (set ADVANCED_RAG_KEY_BUDGET_USD to override), spent $135.44, left $164.56
